# Crime Data

***
**This file will import and aggregate the crime data to create a large table of all crimes done.**  
**It will further use the supportive tables to add more context to the data.**
***

## Aggregate Data

This file will create a final database that can be analysed in visualisation software such as Tableau or PowerBI.

In [1]:
# Import modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

### New Crime ID

Firstly, This file will define a function to create a new Crime ID for the crime data.  
It should identify the police force, the year and month, and be able to store up to 5 digits worth of data (100,000 entries per month).  
It will start by using a sample piece of data.

#### Ingestion

In [2]:
## Import one months worth of data
police_region = 'south-yorkshire'
year_month = '2026-03' # Get the most recent data available

raw_sth_yk = pd.read_csv(f'../Data/Raw/crime-data/{police_region}/{year_month}-{police_region}-street.csv')

raw_sth_yk.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.397019,53.054017,On or near Supermarket,E01019456,Amber Valley 005E,Anti-social behaviour,NaN,NaN
1,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.451708,53.599208,On or near Jack Close Orchard,E01007434,Barnsley 001A,Anti-social behaviour,NaN,NaN
2,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.454483,53.598502,On or near B6428,E01007434,Barnsley 001A,Anti-social behaviour,NaN,NaN
3,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.446274,53.603435,On or near Warren Close,E01007434,Barnsley 001A,Anti-social behaviour,NaN,NaN
4,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.450879,53.602548,On or near Ruston Drive,E01007434,Barnsley 001A,Anti-social behaviour,NaN,NaN


In [3]:
# Import the location lookup table
location_lookup = pd.read_csv('../Data/Processed/location-lookup-table.csv')

location_lookup.head()

,lsoa_code,lsoa_name,lad_code,lad_name,pfa_code,pfa_name
0,E01012000,Hartlepool 007E,E06000001,Hartlepool,E23000013,Cleveland
1,E01011964,Hartlepool 007B,E06000001,Hartlepool,E23000013,Cleveland
2,E01011999,Hartlepool 007D,E06000001,Hartlepool,E23000013,Cleveland
3,E01011967,Hartlepool 007C,E06000001,Hartlepool,E23000013,Cleveland
4,E01011951,Hartlepool 007A,E06000001,Hartlepool,E23000013,Cleveland


#### Cleaning and Validation

Using the cleaning function described in file 01:

In [4]:
def Clean_Crime(raw_crime, dropped_rows):
    # Drop useless columns
    crime = raw_crime[['Crime ID', 'Falls within', 'LSOA code', 'Month', 'Latitude', 'Longitude', 'Crime type']]

    # Rename columns
    crime = crime.rename(columns={
        'Crime ID': 'old_crime_id',
        'Falls within': 'pfa_main',
        'LSOA code': 'lsoa_code',
        'Month': 'date',
        'Latitude': 'latitude',
        'Longitude': 'longitude',
        'Crime type': 'crime_cat'
    })

    # Change month to date
    crime['date'] = pd.to_datetime(crime['date'], format='%Y-%m')

    # Drop duplicate rows
    dropped_rows['duplicates'] = crime.duplicated().sum()
    crime = crime.drop_duplicates()

    # Fill old crime ID with 'No ID' for crimes with no ID
    crime['old_crime_id'] = crime['old_crime_id'].fillna('No ID')

    # Remove rows with null locations
    dropped_rows['No location'] = crime.shape[0] - crime.dropna().shape[0]
    crime = crime.dropna()

    return crime, dropped_rows

In [5]:
dropped_rows = {}
clean_sth_yk, dropped_rows = Clean_Crime(raw_sth_yk, dropped_rows)

In [6]:
clean_sth_yk.head()

,old_crime_id,pfa_main,lsoa_code,date,latitude,longitude,crime_cat
0,No ID,South Yorkshire Police,E01019456,2026-03-01,53.054017,-1.397019,Anti-social behaviour
1,No ID,South Yorkshire Police,E01007434,2026-03-01,53.599208,-1.451708,Anti-social behaviour
2,No ID,South Yorkshire Police,E01007434,2026-03-01,53.598502,-1.454483,Anti-social behaviour
3,No ID,South Yorkshire Police,E01007434,2026-03-01,53.603435,-1.446274,Anti-social behaviour
4,No ID,South Yorkshire Police,E01007434,2026-03-01,53.602548,-1.450879,Anti-social behaviour


In [7]:
for reason in dropped_rows:
    print(f'{reason}: {dropped_rows[reason]}')

duplicates: 658
No location: 668


#### Feature Engineering and Transformation

In [8]:
## Get PFA Crime falls within
clean_sth_yk['falls_within_stripped'] = clean_sth_yk['pfa_main'].str.upper().str.strip('POLICE').str.replace(' ', '_')

clean_sth_yk.head()

,old_crime_id,pfa_main,lsoa_code,date,latitude,longitude,crime_cat,falls_within_stripped
0,No ID,South Yorkshire Police,E01019456,2026-03-01,53.054017,-1.397019,Anti-social behaviour,SOUTH_YORKSHIRE_
1,No ID,South Yorkshire Police,E01007434,2026-03-01,53.599208,-1.451708,Anti-social behaviour,SOUTH_YORKSHIRE_
2,No ID,South Yorkshire Police,E01007434,2026-03-01,53.598502,-1.454483,Anti-social behaviour,SOUTH_YORKSHIRE_
3,No ID,South Yorkshire Police,E01007434,2026-03-01,53.603435,-1.446274,Anti-social behaviour,SOUTH_YORKSHIRE_
4,No ID,South Yorkshire Police,E01007434,2026-03-01,53.602548,-1.450879,Anti-social behaviour,SOUTH_YORKSHIRE_


In [9]:
## Get Year and Month of data
clean_sth_yk['date_str'] = clean_sth_yk['date'].dt.strftime('%Y%m')

clean_sth_yk.head()

,old_crime_id,pfa_main,lsoa_code,date,latitude,longitude,crime_cat,falls_within_stripped,date_str
0,No ID,South Yorkshire Police,E01019456,2026-03-01,53.054017,-1.397019,Anti-social behaviour,SOUTH_YORKSHIRE_,202603
1,No ID,South Yorkshire Police,E01007434,2026-03-01,53.599208,-1.451708,Anti-social behaviour,SOUTH_YORKSHIRE_,202603
2,No ID,South Yorkshire Police,E01007434,2026-03-01,53.598502,-1.454483,Anti-social behaviour,SOUTH_YORKSHIRE_,202603
3,No ID,South Yorkshire Police,E01007434,2026-03-01,53.603435,-1.446274,Anti-social behaviour,SOUTH_YORKSHIRE_,202603
4,No ID,South Yorkshire Police,E01007434,2026-03-01,53.602548,-1.450879,Anti-social behaviour,SOUTH_YORKSHIRE_,202603


In [10]:
## Get 5 digit index of data
clean_sth_yk['index_id'] = (clean_sth_yk.index.astype(str).str.zfill(5))

clean_sth_yk.head()

,old_crime_id,pfa_main,lsoa_code,date,latitude,longitude,crime_cat,falls_within_stripped,date_str,index_id
0,No ID,South Yorkshire Police,E01019456,2026-03-01,53.054017,-1.397019,Anti-social behaviour,SOUTH_YORKSHIRE_,202603,00000
1,No ID,South Yorkshire Police,E01007434,2026-03-01,53.599208,-1.451708,Anti-social behaviour,SOUTH_YORKSHIRE_,202603,00001
2,No ID,South Yorkshire Police,E01007434,2026-03-01,53.598502,-1.454483,Anti-social behaviour,SOUTH_YORKSHIRE_,202603,00002
3,No ID,South Yorkshire Police,E01007434,2026-03-01,53.603435,-1.446274,Anti-social behaviour,SOUTH_YORKSHIRE_,202603,00003
4,No ID,South Yorkshire Police,E01007434,2026-03-01,53.602548,-1.450879,Anti-social behaviour,SOUTH_YORKSHIRE_,202603,00004


In [11]:
# Finally, create a Crime ID using these 3 columns.
clean_sth_yk['crime_id'] = (clean_sth_yk['falls_within_stripped'] + clean_sth_yk['date_str'] + '_' + clean_sth_yk['index_id'])

clean_sth_yk.head()

,old_crime_id,pfa_main,lsoa_code,date,latitude,longitude,crime_cat,falls_within_stripped,date_str,index_id,crime_id
0,No ID,South Yorkshire Police,E01019456,2026-03-01,53.054017,-1.397019,Anti-social behaviour,SOUTH_YORKSHIRE_,202603,00000,SOUTH_YORKSHIRE_202603_00000
1,No ID,South Yorkshire Police,E01007434,2026-03-01,53.599208,-1.451708,Anti-social behaviour,SOUTH_YORKSHIRE_,202603,00001,SOUTH_YORKSHIRE_202603_00001
2,No ID,South Yorkshire Police,E01007434,2026-03-01,53.598502,-1.454483,Anti-social behaviour,SOUTH_YORKSHIRE_,202603,00002,SOUTH_YORKSHIRE_202603_00002
3,No ID,South Yorkshire Police,E01007434,2026-03-01,53.603435,-1.446274,Anti-social behaviour,SOUTH_YORKSHIRE_,202603,00003,SOUTH_YORKSHIRE_202603_00003
4,No ID,South Yorkshire Police,E01007434,2026-03-01,53.602548,-1.450879,Anti-social behaviour,SOUTH_YORKSHIRE_,202603,00004,SOUTH_YORKSHIRE_202603_00004


In [12]:
clean_sth_yk.isnull().sum()

old_crime_id             0
pfa_main                 0
lsoa_code                0
date                     0
latitude                 0
longitude                0
crime_cat                0
falls_within_stripped    0
date_str                 0
index_id                 0
crime_id                 0
dtype: int64

In [13]:
clean_sth_yk.duplicated().sum()

np.int64(0)

Crime ID now created that will not replicate unless there are more than 100,000 entries per month per PFA.  
Function is as follows:

In [14]:
def Finalise_Crime(clean_crime_data):
    ## Get PFA Crime falls within
    clean_crime_data['falls_within_stripped'] = clean_crime_data['pfa_main'].str.upper().str.strip('POLICE').str.replace(' ', '_')

    ## Get Year and Month of data
    clean_crime_data['date_str'] = clean_crime_data['date'].dt.strftime('%Y%m')

    ## Get 5 digit index of data
    clean_crime_data['index_id'] = (clean_crime_data.index.astype(str).str.zfill(5))

    # Finally, create a Crime ID using these 3 columns.
    clean_crime_data['crime_id'] = (clean_crime_data['falls_within_stripped'] + clean_crime_data['date_str'] + '_' + clean_crime_data['index_id'])

    # Clear columns and set an order
    final_crime_data = clean_crime_data[['crime_id', 'date', 'pfa_main', 'lsoa_code', 'latitude', 'longitude', 'crime_cat']]

    return final_crime_data

***
***
### Crime Data Aggregation

The end goal of this section is to create the general functions for importing all the data.

It will require a loop that names each individual police force, then each date for the data.  
within the loop, it should import that specific data.  
then clean the data, and check for issues with that.  
then create the same final database for the data, as is shown above.  
Finally, it should append the data to a large database holding all the data, and loop to the next entry.  

In [15]:
# Loop through each police force, then its dates

police_regions = ['merseyside', 'south-yorkshire', 'west-midlands', 'west-yorkshire']
years = ['2023', '2024', '2025', '2026']
months = ['01','02','03','04','05','06','07','08','09','10','11','12']

# Final DataFrame
aggregated_crime_data = pd.DataFrame(columns=['crime_id', 'date', 'pfa_main', 'lsoa_code', 'latitude', 'longitude', 'crime_cat'])

skipped_data = []
dropped_rows = {}
total_rows_dropped = 0

for police_region in police_regions:
    for year in years:
        for month in months:
            file_path = f'../Data/Raw/crime-data/{police_region}/{year}-{month}-{police_region}-street.csv'

            try:
                raw_crime_data = pd.read_csv(file_path)
                print(f'imported {police_region} {year} {month} data')
            except FileNotFoundError:
                print(f'Missing file: {file_path}, skipping...')
                skipped_data.append(f'{police_region}-{year}/{month}, reason: importing')
                continue

            clean_crime_data, dropped_rows = Clean_Crime(raw_crime_data, dropped_rows)

            print(f'Cleaned {police_region} {year} {month} data')
            total_rows_dropped += dropped_rows['No location'] # Add dropped rows with data to total_rows_dropped

            final_crime_data = Finalise_Crime(clean_crime_data)

            print(f'Finalised {police_region} {year} {month} data') # No rows dropped in this function

            # Check if clean
            if final_crime_data.duplicated().sum() != 0:
                # Drop duplicate rows and add them to the total rows dropped
                total_rows_dropped += final_crime_data.duplicated().sum()
                final_crime_data = final_crime_data.drop_duplicates()

                print(f'Duplicate values in final data, dropped')

            if final_crime_data.isnull().sum().sum() != 0:
                # Drop rows with nulls in
                total_rows_dropped += final_crime_data.shape[0] - final_crime_data.dropna().shape[0]
                final_crime_data = final_crime_data.dropna()

                print(f'Null values in final data, dropped')
            
            
            # Append to final data
            aggregated_crime_data = pd.concat([aggregated_crime_data, final_crime_data], ignore_index=True)
            print(f'Added {police_region} {year} {month} data to end data')

print(f'\nTotal rows in final Data: {aggregated_crime_data.shape[0]}')
print(f'Total rows dropped in cleaning and aggregation process: {total_rows_dropped}')
print(f'Percentage of data lost: {(total_rows_dropped/aggregated_crime_data.shape[0])*100}%')


Missing file: ../Data/Raw/crime-data/merseyside/2023-01-merseyside-street.csv, skipping...
Missing file: ../Data/Raw/crime-data/merseyside/2023-02-merseyside-street.csv, skipping...
Missing file: ../Data/Raw/crime-data/merseyside/2023-03-merseyside-street.csv, skipping...
imported merseyside 2023 04 data
Cleaned merseyside 2023 04 data
Finalised merseyside 2023 04 data
Added merseyside 2023 04 data to end data
imported merseyside 2023 05 data
Cleaned merseyside 2023 05 data


C:\Users\sam\AppData\Local\Temp\ipykernel_23612\1085613425.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  aggregated_crime_data = pd.concat([aggregated_crime_data, final_crime_data], ignore_index=True)


Finalised merseyside 2023 05 data
Added merseyside 2023 05 data to end data
imported merseyside 2023 06 data
Cleaned merseyside 2023 06 data
Finalised merseyside 2023 06 data
Added merseyside 2023 06 data to end data
imported merseyside 2023 07 data
Cleaned merseyside 2023 07 data
Finalised merseyside 2023 07 data
Added merseyside 2023 07 data to end data
imported merseyside 2023 08 data
Cleaned merseyside 2023 08 data
Finalised merseyside 2023 08 data
Added merseyside 2023 08 data to end data
imported merseyside 2023 09 data
Cleaned merseyside 2023 09 data
Finalised merseyside 2023 09 data
Added merseyside 2023 09 data to end data
imported merseyside 2023 10 data
Cleaned merseyside 2023 10 data
Finalised merseyside 2023 10 data
Added merseyside 2023 10 data to end data
imported merseyside 2023 11 data
Cleaned merseyside 2023 11 data
Finalised merseyside 2023 11 data
Added merseyside 2023 11 data to end data
imported merseyside 2023 12 data
Cleaned merseyside 2023 12 data
Finalised mer

***
**All Crime data imported successfully**

Just over 1% of data lost in the process.
***

In [16]:
aggregated_crime_data.sample(10)

,crime_id,date,pfa_main,lsoa_code,latitude,longitude,crime_cat
1978973,WEST_YORKSHIRE_202305_18565,2023-05-01,West Yorkshire Police,E01011433,53.809440,-1.509198,Other theft
2748955,WEST_YORKSHIRE_202512_06407,2025-12-01,West Yorkshire Police,E01010998,53.726466,-1.893825,Violence and sexual offences
735002,SOUTH_YORKSHIRE_202412_04449,2024-12-01,South Yorkshire Police,E01034242,53.496212,-1.129486,Vehicle crime
847933,SOUTH_YORKSHIRE_202508_13353,2025-08-01,South Yorkshire Police,E01033263,53.388625,-1.475783,Theft from the person
1333727,WEST_MIDLANDS_202405_20401,2024-05-01,West Midlands Police,E01010097,52.548495,-2.037163,Violence and sexual offences
2553631,WEST_YORKSHIRE_202504_11943,2025-04-01,West Yorkshire Police,E01011087,53.614082,-1.815505,Public order
509337,SOUTH_YORKSHIRE_202307_13266,2023-07-01,South Yorkshire Police,E01007978,53.358406,-1.469843,Burglary
2594883,WEST_YORKSHIRE_202506_02210,2025-06-01,West Yorkshire Police,E01010674,53.820053,-1.809772,Violence and sexual offences
2190102,WEST_YORKSHIRE_202401_17006,2024-01-01,West Yorkshire Police,E01011312,53.775253,-1.544208,Shoplifting
2375611,WEST_YORKSHIRE_202409_00807,2024-09-01,West Yorkshire Police,E01010718,53.849431,-1.931440,Criminal damage and arson


## Individual Crime Data

#### Location Lookup

In [17]:
crime_location = pd.merge(aggregated_crime_data, location_lookup, how='left', on=['lsoa_code'])

crime_location.sample(10)

,crime_id,date,pfa_main,lsoa_code,latitude,longitude,crime_cat,lsoa_name,lad_code,lad_name,pfa_code,pfa_name
2087288,WEST_YORKSHIRE_202309_14766,2023-09-01,West Yorkshire Police,E01011354,53.826759,-1.555226,Criminal damage and arson,Leeds 038A,E08000035,Leeds,E23000010,West Yorkshire
1009704,WEST_MIDLANDS_202306_10916,2023-06-01,West Midlands Police,E01009109,52.400778,-1.920147,Violence and sexual offences,Birmingham 123B,E08000025,Birmingham,E23000014,West Midlands
1434123,WEST_MIDLANDS_202409_01424,2024-09-01,West Midlands Police,E01009098,52.523477,-1.811891,Violence and sexual offences,Birmingham 025C,E08000025,Birmingham,E23000014,West Midlands
111715,MERSEYSIDE_202312_02470,2023-12-01,Merseyside Police,E01006764,53.425649,-2.932156,Criminal damage and arson,Liverpool 020D,E08000012,Liverpool,E23000004,Merseyside
436728,MERSEYSIDE_202602_08969,2026-02-01,Merseyside Police,E01006912,53.447650,-2.734186,Drugs,St. Helens 024E,E08000013,St. Helens,E23000004,Merseyside
2144075,WEST_YORKSHIRE_202311_18338,2023-11-01,West Yorkshire Police,E01011474,53.765105,-1.525659,Criminal damage and arson,Leeds 092D,E08000035,Leeds,E23000010,West Yorkshire
2041201,WEST_YORKSHIRE_202307_23060,2023-07-01,West Yorkshire Police,E01033010,53.804869,-1.546909,Vehicle crime,Leeds 111B,E08000035,Leeds,E23000010,West Yorkshire
879111,SOUTH_YORKSHIRE_202511_04025,2025-11-01,South Yorkshire Police,E01007655,53.525120,-1.134460,Shoplifting,Doncaster 022F,E08000017,Doncaster,E23000011,South Yorkshire
1985322,WEST_YORKSHIRE_202305_25065,2023-05-01,West Yorkshire Police,E01033010,53.795751,-1.541949,Robbery,Leeds 111B,E08000035,Leeds,E23000010,West Yorkshire
908072,SOUTH_YORKSHIRE_202601_07989,2026-01-01,South Yorkshire Police,E01008118,53.423837,-1.482799,Other crime,Sheffield 009E,E08000039,Sheffield,E23000011,South Yorkshire


In [18]:
crime_location.shape

(2830431, 12)

#### Population

In [19]:
# Import population data

population = pd.read_csv('../Data/Processed/population.csv')

In [20]:
crime_location['year'] = crime_location['date'].dt.year

In [21]:
crime_location_pop = pd.merge(crime_location, population, how='left', on=['lsoa_code', 'year'])

crime_location_pop.head()

,crime_id,date,pfa_main,lsoa_code,latitude,longitude,crime_cat,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,year,population
0,MERSEYSIDE_202304_00000,2023-04-01,Merseyside Police,E01018537,53.301368,-3.089063,Anti-social behaviour,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,2018.0
1,MERSEYSIDE_202304_00001,2023-04-01,Merseyside Police,E01018537,53.315654,-3.074956,Vehicle crime,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,2018.0
2,MERSEYSIDE_202304_00002,2023-04-01,Merseyside Police,E01018570,53.300746,-2.960659,Other crime,Cheshire West and Chester 004C,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,1619.0
3,MERSEYSIDE_202304_00003,2023-04-01,Merseyside Police,E01012393,53.389101,-2.746819,Violence and sexual offences,Halton 001B,E06000006,Halton,E23000006,Cheshire,2023,2948.0
4,MERSEYSIDE_202304_00004,2023-04-01,Merseyside Police,E01012376,53.377272,-2.756590,Anti-social behaviour,Halton 002C,E06000006,Halton,E23000006,Cheshire,2023,2822.0


In [22]:
crime_location_pop.shape

(2830431, 14)

#### Deprivation

In [23]:
# Import Deprivation

deprivation = pd.read_csv('../Data/Processed/deprivation.csv')

In [24]:
crime_loc_pop_depr = pd.merge(crime_location_pop, deprivation, how='left', on='lsoa_code')

crime_loc_pop_depr.sample(10)

,crime_id,date,pfa_main,lsoa_code,latitude,longitude,crime_cat,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,year,population,imd_score,incm_score,empl_score,edcn_score,hous_score
2725299,WEST_YORKSHIRE_202511_05753,2025-11-01,West Yorkshire Police,E01033690,53.793982,-1.752296,Violence and sexual offences,Bradford 065C,E08000032,Bradford,E23000010,West Yorkshire,2025,2568.0,49.203,0.279,0.218,29.742,37.241
2205903,WEST_YORKSHIRE_202402_09542,2024-02-01,West Yorkshire Police,E01011093,53.651003,-1.762861,Violence and sexual offences,Kirklees 035A,E08000034,Kirklees,E23000010,West Yorkshire,2024,1449.0,59.536,0.586,0.364,53.634,19.169
1487879,WEST_MIDLANDS_202410_27692,2024-10-01,West Midlands Police,E01010445,52.589184,-2.098853,Criminal damage and arson,Wolverhampton 018D,E08000031,Wolverhampton,E23000014,West Midlands,2024,1507.0,50.734,0.561,0.299,48.192,22.871
2824591,WEST_YORKSHIRE_202603_18044,2026-03-01,West Yorkshire Police,E01011469,53.757864,-1.522009,Other crime,Leeds 094B,E08000035,Leeds,E23000010,West Yorkshire,2026,1439.0,21.124,0.187,0.137,33.982,15.967
2298473,WEST_YORKSHIRE_202406_03351,2024-06-01,West Yorkshire Police,E01010833,53.792219,-1.778871,Vehicle crime,Bradford 041C,E08000032,Bradford,E23000010,West Yorkshire,2024,1789.0,62.595,0.577,0.300,79.825,30.778
432278,MERSEYSIDE_202602_04374,2026-02-01,Merseyside Police,E01006551,53.387937,-2.938278,Violence and sexual offences,Liverpool 048E,E08000012,Liverpool,E23000004,Merseyside,2026,1322.0,43.096,0.416,0.311,24.031,17.395
2414278,WEST_YORKSHIRE_202410_15096,2024-10-01,West Yorkshire Police,E01011431,53.809934,-1.504682,Burglary,Leeds 047D,E08000035,Leeds,E23000010,West Yorkshire,2024,2182.0,55.757,0.539,0.297,57.981,17.091
609209,SOUTH_YORKSHIRE_202403_04233,2024-03-01,South Yorkshire Police,E01007529,53.517618,-1.151248,Violence and sexual offences,Doncaster 022A,E08000017,Doncaster,E23000011,South Yorkshire,2024,2175.0,72.128,0.522,0.342,93.862,18.895
69648,MERSEYSIDE_202308_12627,2023-08-01,Merseyside Police,E01007162,53.384107,-3.074861,Public order,Wirral 019C,E08000015,Wirral,E23000004,Merseyside,2023,1520.0,56.129,0.554,0.338,60.260,11.675
974207,WEST_MIDLANDS_202305_05453,2023-05-01,West Midlands Police,E01009311,52.483170,-1.792376,Violence and sexual offences,Birmingham 054E,E08000025,Birmingham,E23000014,West Midlands,2023,1635.0,59.195,0.591,0.309,62.548,20.722


In [25]:
crime_loc_pop_depr.shape

(2830431, 19)

#### Crime Severity

In [26]:
# Import Crime severity data

severity = pd.read_csv('../Data/Processed/crime-severity.csv')

In [27]:
severity.head()
severity = severity.drop(columns='Unnamed: 0')

In [28]:
# Modify crime cat to be lower case
crime_loc_pop_depr['crime_cat'] = crime_loc_pop_depr['crime_cat'].str.lower()

In [29]:
crime_loc_pop_depr.head()

,crime_id,date,pfa_main,lsoa_code,latitude,longitude,crime_cat,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,year,population,imd_score,incm_score,empl_score,edcn_score,hous_score
0,MERSEYSIDE_202304_00000,2023-04-01,Merseyside Police,E01018537,53.301368,-3.089063,anti-social behaviour,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,2018.0,4.137,0.050,0.049,0.889,18.163
1,MERSEYSIDE_202304_00001,2023-04-01,Merseyside Police,E01018537,53.315654,-3.074956,vehicle crime,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,2018.0,4.137,0.050,0.049,0.889,18.163
2,MERSEYSIDE_202304_00002,2023-04-01,Merseyside Police,E01018570,53.300746,-2.960659,other crime,Cheshire West and Chester 004C,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,1619.0,10.524,0.087,0.060,4.840,26.361
3,MERSEYSIDE_202304_00003,2023-04-01,Merseyside Police,E01012393,53.389101,-2.746819,violence and sexual offences,Halton 001B,E06000006,Halton,E23000006,Cheshire,2023,2948.0,4.483,0.051,0.051,2.076,16.072
4,MERSEYSIDE_202304_00004,2023-04-01,Merseyside Police,E01012376,53.377272,-2.756590,anti-social behaviour,Halton 002C,E06000006,Halton,E23000006,Cheshire,2023,2822.0,5.531,0.053,0.053,3.032,21.859


In [30]:
crime_loc_pop_depr_sev = pd.merge(crime_loc_pop_depr, severity, how='left', on='crime_cat')

crime_loc_pop_depr_sev.sample(10)

,crime_id,date,pfa_main,lsoa_code,latitude,longitude,crime_cat,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,year,population,imd_score,incm_score,empl_score,edcn_score,hous_score,avg_weight
643039,SOUTH_YORKSHIRE_202405_11094,2024-05-01,South Yorkshire Police,E01033269,53.392656,-1.434223,criminal damage and arson,Sheffield 022G,E08000039,Sheffield,E23000011,South Yorkshire,2024,1986.0,64.261,0.726,0.353,68.768,23.644,19.0
163449,MERSEYSIDE_202404_07417,2024-04-01,Merseyside Police,E01006922,53.600045,-3.052803,violence and sexual offences,Sefton 012B,E08000014,Sefton,E23000004,Merseyside,2024,1496.0,8.469,0.049,0.095,6.071,16.884,709.0
2744712,WEST_YORKSHIRE_202512_02128,2025-12-01,West Yorkshire Police,E01010676,53.809075,-1.775973,violence and sexual offences,Bradford 034A,E08000032,Bradford,E23000010,West Yorkshire,2025,2526.0,57.312,0.564,0.263,71.983,26.875,709.0
1848776,WEST_MIDLANDS_202512_14858,2025-12-01,West Midlands Police,E01009830,52.489204,-2.165652,shoplifting,Dudley 019F,E08000027,Dudley,E23000014,West Midlands,2025,1698.0,5.231,0.076,0.070,7.242,9.955,13.0
723218,SOUTH_YORKSHIRE_202411_06566,2024-11-01,South Yorkshire Police,E01007728,53.453150,-1.362169,violence and sexual offences,Rotherham 009A,E08000018,Rotherham,E23000011,South Yorkshire,2024,1113.0,19.635,0.193,0.141,23.006,16.703,709.0
1320487,WEST_MIDLANDS_202405_06895,2024-05-01,West Midlands Police,E01008984,52.456345,-1.907141,vehicle crime,Birmingham 074A,E08000025,Birmingham,E23000014,West Midlands,2024,1908.0,46.851,0.480,0.297,30.847,23.167,41.0
2053370,WEST_YORKSHIRE_202308_07428,2023-08-01,West Yorkshire Police,E01034582,53.722971,-1.859493,shoplifting,Calderdale 008G,E08000033,Calderdale,E23000010,West Yorkshire,2023,1660.0,60.681,0.456,0.340,30.917,23.866,13.0
890154,SOUTH_YORKSHIRE_202512_02254,2025-12-01,South Yorkshire Police,E01007639,53.611537,-0.963689,possession of weapons,Doncaster 003D,E08000017,Doncaster,E23000011,South Yorkshire,2025,2402.5,33.201,0.284,0.180,30.756,27.069,75.0
25169,MERSEYSIDE_202305_11402,2023-05-01,Merseyside Police,E01006877,53.454588,-2.748868,anti-social behaviour,St. Helens 012C,E08000013,St. Helens,E23000004,Merseyside,2023,1613.0,60.397,0.484,0.366,40.188,12.786,NaN
1023848,WEST_MIDLANDS_202306_25334,2023-06-01,West Midlands Police,E01010211,52.397137,-1.833224,drugs,Solihull 023C,E08000029,Solihull,E23000014,West Midlands,2023,1406.0,22.810,0.262,0.191,20.331,10.413,9.0


In [31]:
crime_loc_pop_depr_sev.shape

(2830431, 20)

***
Only Anti-social behaviour is null crime category.  
The crime severity weighting is based off of the severity of court rulings coming from crimes. As ASB often goes unpunished, it does not have a crime severity weighting score.  
**assumption:** Crime severity for ASB is 1. This is a low enough number to barely weigh in to crime severity, but does not leave it unnoticed.
***

In [32]:
crime_loc_pop_depr_sev['avg_weight'] = crime_loc_pop_depr_sev['avg_weight'].fillna(1)

In [33]:
crime_loc_pop_depr_sev.head()

,crime_id,date,pfa_main,lsoa_code,latitude,longitude,crime_cat,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,year,population,imd_score,incm_score,empl_score,edcn_score,hous_score,avg_weight
0,MERSEYSIDE_202304_00000,2023-04-01,Merseyside Police,E01018537,53.301368,-3.089063,anti-social behaviour,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,2018.0,4.137,0.050,0.049,0.889,18.163,1.0
1,MERSEYSIDE_202304_00001,2023-04-01,Merseyside Police,E01018537,53.315654,-3.074956,vehicle crime,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,2018.0,4.137,0.050,0.049,0.889,18.163,41.0
2,MERSEYSIDE_202304_00002,2023-04-01,Merseyside Police,E01018570,53.300746,-2.960659,other crime,Cheshire West and Chester 004C,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,1619.0,10.524,0.087,0.060,4.840,26.361,86.0
3,MERSEYSIDE_202304_00003,2023-04-01,Merseyside Police,E01012393,53.389101,-2.746819,violence and sexual offences,Halton 001B,E06000006,Halton,E23000006,Cheshire,2023,2948.0,4.483,0.051,0.051,2.076,16.072,709.0
4,MERSEYSIDE_202304_00004,2023-04-01,Merseyside Police,E01012376,53.377272,-2.756590,anti-social behaviour,Halton 002C,E06000006,Halton,E23000006,Cheshire,2023,2822.0,5.531,0.053,0.053,3.032,21.859,1.0


In [34]:
crime_final = crime_loc_pop_depr_sev[[
    'crime_id', 
    'date', 
    'year', 
    'pfa_main',
    'pfa_code', 
    'pfa_name', 
    'latitude', 
    'longitude', 
    'lsoa_code', 
    'lsoa_name',
    'lad_code',
    'lad_name',
    'crime_cat',
    'avg_weight',
    'population',
    'imd_score',
    'incm_score',
    'empl_score',
    'edcn_score',
    'hous_score'
]]

crime_final.head()

,crime_id,date,year,pfa_main,pfa_code,pfa_name,latitude,longitude,lsoa_code,lsoa_name,lad_code,lad_name,crime_cat,avg_weight,population,imd_score,incm_score,empl_score,edcn_score,hous_score
0,MERSEYSIDE_202304_00000,2023-04-01,2023,Merseyside Police,E23000006,Cheshire,53.301368,-3.089063,E01018537,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,anti-social behaviour,1.0,2018.0,4.137,0.050,0.049,0.889,18.163
1,MERSEYSIDE_202304_00001,2023-04-01,2023,Merseyside Police,E23000006,Cheshire,53.315654,-3.074956,E01018537,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,vehicle crime,41.0,2018.0,4.137,0.050,0.049,0.889,18.163
2,MERSEYSIDE_202304_00002,2023-04-01,2023,Merseyside Police,E23000006,Cheshire,53.300746,-2.960659,E01018570,Cheshire West and Chester 004C,E06000050,Cheshire West and Chester,other crime,86.0,1619.0,10.524,0.087,0.060,4.840,26.361
3,MERSEYSIDE_202304_00003,2023-04-01,2023,Merseyside Police,E23000006,Cheshire,53.389101,-2.746819,E01012393,Halton 001B,E06000006,Halton,violence and sexual offences,709.0,2948.0,4.483,0.051,0.051,2.076,16.072
4,MERSEYSIDE_202304_00004,2023-04-01,2023,Merseyside Police,E23000006,Cheshire,53.377272,-2.756590,E01012376,Halton 002C,E06000006,Halton,anti-social behaviour,1.0,2822.0,5.531,0.053,0.053,3.032,21.859


***
***
### Final Database Check and Export

In [35]:
# Check cleanliness
crime_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2830431 entries, 0 to 2830430
Data columns (total 20 columns):
 #   Column      Dtype         
---  ------      -----         
 0   crime_id    object        
 1   date        datetime64[ns]
 2   year        int32         
 3   pfa_main    object        
 4   pfa_code    object        
 5   pfa_name    object        
 6   latitude    float64       
 7   longitude   float64       
 8   lsoa_code   object        
 9   lsoa_name   object        
 10  lad_code    object        
 11  lad_name    object        
 12  crime_cat   object        
 13  avg_weight  float64       
 14  population  float64       
 15  imd_score   float64       
 16  incm_score  float64       
 17  empl_score  float64       
 18  edcn_score  float64       
 19  hous_score  float64       
dtypes: datetime64[ns](1), float64(9), int32(1), object(9)
memory usage: 421.1+ MB


In [36]:
crime_final.isnull().sum()

crime_id         0
date             0
year             0
pfa_main         0
pfa_code      7569
pfa_name      7569
latitude         0
longitude        0
lsoa_code        0
lsoa_name     7569
lad_code      7569
lad_name      7569
crime_cat        0
avg_weight       0
population    7569
imd_score     7573
incm_score    7573
empl_score    7573
edcn_score    7573
hous_score    7573
dtype: int64

In [37]:
crime_final[crime_final['pfa_code'].isnull()]['lsoa_code'].value_counts()

lsoa_code
E01007645    469
E01009642    450
E01010521    369
E01010994    346
E01011916    292
            ... 
E01007859      5
E01033766      4
E01033749      3
E01024618      1
E01024929      1
Name: count, Length: 104, dtype: int64

***
For some reason the location lookup table seems to be missing some lsoa codes - likely due to a mismatch in the year of intake.  
Unfortunately it is too late in the project to fix this, so the rows will have to be dropped.
***

In [38]:
total_rows_dropped += crime_final.shape[0] - crime_final.dropna().shape[0]
crime_final = crime_final.dropna()

print(f'\nTotal rows in final Data: {crime_final.shape[0]}')
print(f'Total rows dropped in cleaning and aggregation process: {total_rows_dropped}')
print(f'Percentage of data lost: {(total_rows_dropped/crime_final.shape[0])*100}%')


Total rows in final Data: 2822858
Total rows dropped in cleaning and aggregation process: 41197
Percentage of data lost: 1.4594074515969275%


In [39]:
crime_final.duplicated().sum()

np.int64(0)

In [40]:
crime_final = crime_final.rename(columns={'avg_weight':'crime_sev'})

***
**Final data is now fully cclean and ready to export**
***

In [41]:
# Final check
crime_final.sample(10)

,crime_id,date,year,pfa_main,pfa_code,pfa_name,latitude,longitude,lsoa_code,lsoa_name,lad_code,lad_name,crime_cat,crime_sev,population,imd_score,incm_score,empl_score,edcn_score,hous_score
1386404,WEST_MIDLANDS_202407_13295,2024-07-01,2024,West Midlands Police,E23000014,West Midlands,52.479541,-1.898111,E01033620,Birmingham 138A,E08000025,Birmingham,violence and sexual offences,709.0,1503.0,28.579,0.162,0.156,13.372,19.783
1593420,WEST_MIDLANDS_202503_03497,2025-03-01,2025,West Midlands Police,E23000014,West Midlands,52.492875,-1.925003,E01009348,Birmingham 047A,E08000025,Birmingham,violence and sexual offences,709.0,2185.0,42.877,0.553,0.277,42.414,25.907
2656984,WEST_YORKSHIRE_202508_12204,2025-08-01,2025,West Yorkshire Police,E23000010,West Yorkshire,53.636842,-1.751628,E01011010,Kirklees 044C,E08000034,Kirklees,anti-social behaviour,1.0,1610.5,44.091,0.458,0.235,54.372,17.584
2519191,WEST_YORKSHIRE_202503_02680,2025-03-01,2025,West Yorkshire Police,E23000010,West Yorkshire,53.808260,-1.750253,E01010830,Bradford 035C,E08000032,Bradford,violence and sexual offences,709.0,2709.0,57.206,0.516,0.235,70.939,22.169
924044,SOUTH_YORKSHIRE_202603_00091,2026-03-01,2026,South Yorkshire Police,E23000011,South Yorkshire,53.587603,-1.453475,E01007437,Barnsley 002C,E08000038,Barnsley,anti-social behaviour,1.0,1466.0,32.021,0.293,0.210,34.828,17.662
1788372,WEST_MIDLANDS_202510_06836,2025-10-01,2025,West Midlands Police,E23000014,West Midlands,52.459602,-1.888443,E01009374,Birmingham 084B,E08000025,Birmingham,violence and sexual offences,709.0,1341.0,55.369,0.643,0.315,43.364,28.319
2044566,WEST_YORKSHIRE_202307_26545,2023-07-01,2023,West Yorkshire Police,E23000010,West Yorkshire,53.677574,-1.526945,E01011900,Wakefield 026B,E08000036,Wakefield,violence and sexual offences,709.0,1583.0,40.144,0.390,0.239,42.828,8.182
1929688,WEST_MIDLANDS_202603_23697,2026-03-01,2026,West Midlands Police,E23000014,West Midlands,52.585143,-1.989906,E01010363,Walsall 026D,E08000030,Walsall,anti-social behaviour,1.0,3118.0,39.714,0.461,0.232,51.961,21.540
2637648,WEST_YORKSHIRE_202507_19577,2025-07-01,2025,West Yorkshire Police,E23000010,West Yorkshire,53.784419,-1.560621,E01033013,Leeds 082E,E08000035,Leeds,violence and sexual offences,709.0,2096.0,34.029,0.404,0.196,25.112,17.475
2439974,WEST_YORKSHIRE_202411_15286,2024-11-01,2024,West Yorkshire Police,E23000010,West Yorkshire,53.815563,-1.437939,E01011407,Leeds 057B,E08000035,Leeds,drugs,9.0,1396.0,19.316,0.181,0.117,26.963,4.505


In [42]:
# Export to csv file

## Output Final Table to csv
crime_final.to_csv('../Data/Output/crime-final.csv', index=False)

print(f'crime_final.csv File successfully created: {Path('../Data/Output/crime-final.csv').exists()}')

crime_final.csv File successfully created: True


## LSOA Grained Data

Grained / month  
Final Columns:  
Location columns || PFA main | Date | Population | Crime Count | Avg Crime per 1000 | Avg Crime Severity || deprivation stats ||

#### Crime Severity
As this is grained by individual crime, this needs to be merged before grouping

In [43]:
## Crime Severity Data already imported
# Lower case crime category

aggregated_crime_data['crime_cat'] = aggregated_crime_data['crime_cat'].str.lower()

In [44]:
# Merge on crime cat

severity_agg = pd.merge(aggregated_crime_data,severity,how='left',on='crime_cat')

severity_agg.sample(5)

,crime_id,date,pfa_main,lsoa_code,latitude,longitude,crime_cat,avg_weight
1194144,WEST_MIDLANDS_202312_16960,2023-12-01,West Midlands Police,E01009829,52.488200,-2.163851,violence and sexual offences,709.0
2334803,WEST_YORKSHIRE_202407_14023,2024-07-01,West Yorkshire Police,E01011278,53.854191,-1.686419,vehicle crime,41.0
1740535,WEST_MIDLANDS_202508_11926,2025-08-01,West Midlands Police,E01033620,52.482804,-1.897764,shoplifting,13.0
127547,MERSEYSIDE_202401_06808,2024-01-01,Merseyside Police,E01007035,53.659840,-2.963950,violence and sexual offences,709.0
2418842,WEST_YORKSHIRE_202410_19753,2024-10-01,West Yorkshire Police,E01011520,53.758238,-1.633335,anti-social behaviour,NaN


In [45]:
# Assume anti-socila behaviour to have severity of 1
severity_agg['avg_weight'] = severity_agg['avg_weight'].fillna(1)

#### Group by LSOA

In [46]:
# Formulate final table
by_lsoa = (
    severity_agg
    .groupby(['lsoa_code', 'date'], as_index=False)
    .agg(
        pfa_main=('pfa_main', lambda x: x.mode().iloc[0]),
        crime_count=('crime_id', 'nunique'),
        avg_crime_sev=('avg_weight', 'mean')
    )
)

by_lsoa.sample(7)

,lsoa_code,date,pfa_main,crime_count,avg_crime_sev
168996,E01033624,2026-01-01,West Midlands Police,7,519.642857
138561,E01011203,2024-08-01,West Yorkshire Police,15,510.866667
100778,E01010121,2023-06-01,West Midlands Police,11,225.727273
61455,E01008943,2023-12-01,West Midlands Police,19,321.842105
23046,E01007097,2024-01-01,Merseyside Police,2,244.750000
5879,E01006586,2024-03-01,Merseyside Police,17,487.970588
174467,E01034842,2025-01-01,South Yorkshire Police,57,293.991228


In [47]:
by_lsoa.shape

(176723, 5)

#### Location Lookup

In [48]:
location_lookup.head()

,lsoa_code,lsoa_name,lad_code,lad_name,pfa_code,pfa_name
0,E01012000,Hartlepool 007E,E06000001,Hartlepool,E23000013,Cleveland
1,E01011964,Hartlepool 007B,E06000001,Hartlepool,E23000013,Cleveland
2,E01011999,Hartlepool 007D,E06000001,Hartlepool,E23000013,Cleveland
3,E01011967,Hartlepool 007C,E06000001,Hartlepool,E23000013,Cleveland
4,E01011951,Hartlepool 007A,E06000001,Hartlepool,E23000013,Cleveland


In [49]:
lsoa_location = pd.merge(by_lsoa, location_lookup, how='left', on='lsoa_code')

lsoa_location.sample(7)

,lsoa_code,date,pfa_main,crime_count,avg_crime_sev,lsoa_name,lad_code,lad_name,pfa_code,pfa_name
63615,E01009007,2024-06-01,West Midlands Police,54,340.722222,Birmingham 023D,E08000025,Birmingham,E23000014,West Midlands
15551,E01006877,2023-11-01,Merseyside Police,74,379.439189,St. Helens 012C,E08000013,St. Helens,E23000004,Merseyside
112609,E01010463,2024-05-01,West Midlands Police,27,368.574074,Wolverhampton 020A,E08000031,Wolverhampton,E23000014,West Midlands
140582,E01011260,2025-11-01,West Yorkshire Police,8,452.625000,Kirklees 023E,E08000034,Kirklees,E23000010,West Yorkshire
30796,E01007323,2025-04-01,South Yorkshire Police,16,498.718750,Barnsley 007A,E08000038,Barnsley,E23000011,South Yorkshire
69632,E01009197,2025-01-01,West Midlands Police,16,437.000000,Birmingham 058C,E08000025,Birmingham,E23000014,West Midlands
104871,E01010239,2025-08-01,West Midlands Police,4,536.500000,Solihull 002C,E08000029,Solihull,E23000014,West Midlands


In [50]:
lsoa_location.shape

(176723, 10)

#### Population

In [51]:
lsoa_location['year'] = lsoa_location['date'].dt.year

lsoa_loc_pop = pd.merge(lsoa_location, population, how='left',on=['lsoa_code', 'year'])

lsoa_loc_pop.sample(7)

,lsoa_code,date,pfa_main,crime_count,avg_crime_sev,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,year,population
116639,E01010577,2023-07-01,West Yorkshire Police,8,448.000000,Bradford 013D,E08000032,Bradford,E23000010,West Yorkshire,2023,1581.0
108865,E01010355,2025-05-01,West Midlands Police,3,250.333333,Walsall 039B,E08000030,Walsall,E23000014,West Midlands,2025,1513.5
18577,E01006964,2024-12-01,Merseyside Police,12,276.541667,Sefton 036A,E08000014,Sefton,E23000004,Merseyside,2024,1614.0
65496,E01009065,2025-08-01,West Midlands Police,16,457.937500,Birmingham 075A,E08000025,Birmingham,E23000014,West Midlands,2025,1660.0
174429,E01034841,2024-11-01,Merseyside Police,11,426.272727,Wirral 036G,E08000015,Wirral,E23000004,Merseyside,2024,1497.0
86513,E01009711,2025-11-01,West Midlands Police,12,381.958333,Coventry 013D,E08000026,Coventry,E23000014,West Midlands,2025,2112.5
135610,E01011117,2023-06-01,West Yorkshire Police,8,503.125000,Kirklees 057F,E08000034,Kirklees,E23000010,West Yorkshire,2023,1654.0


In [52]:
lsoa_loc_pop.shape

(176723, 12)

#### Deprivation

In [53]:
lsoa_loc_pop_depr = pd.merge(lsoa_loc_pop, deprivation, how='left', on='lsoa_code')

lsoa_loc_pop_depr.sample(7)

,lsoa_code,date,pfa_main,crime_count,avg_crime_sev,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,year,population,imd_score,incm_score,empl_score,edcn_score,hous_score
170929,E01033894,2024-12-01,West Midlands Police,11,339.954545,Walsall 013F,E08000030,Walsall,E23000014,West Midlands,2024,2245.0,56.427,0.657,0.326,51.430,14.264
134806,E01011094,2025-04-01,West Yorkshire Police,17,354.941176,Kirklees 035B,E08000034,Kirklees,E23000010,West Yorkshire,2025,1579.5,58.090,0.562,0.363,50.755,20.805
73491,E01009309,2025-10-01,West Midlands Police,26,440.038462,Birmingham 045E,E08000025,Birmingham,E23000014,West Midlands,2025,1900.5,53.067,0.517,0.304,46.409,24.292
164023,E01019770,2026-02-01,South Yorkshire Police,1,1.000000,North East Derbyshire 005A,E07000038,North East Derbyshire,E23000018,Derbyshire,2026,2044.0,13.631,0.136,0.074,9.797,31.986
169397,E01033648,2024-04-01,West Midlands Police,40,484.987500,Birmingham 084F,E08000025,Birmingham,E23000014,West Midlands,2024,2764.0,60.930,0.692,0.319,50.944,31.541
61677,E01008949,2024-06-01,West Midlands Police,4,45.250000,Birmingham 122E,E08000025,Birmingham,E23000014,West Midlands,2024,1208.0,8.711,0.114,0.071,11.347,14.692
88923,E01009781,2024-12-01,West Midlands Police,12,496.916667,Dudley 007A,E08000027,Dudley,E23000014,West Midlands,2024,1922.0,26.139,0.284,0.200,30.529,13.975


In [54]:
lsoa_loc_pop_depr.shape

(176723, 17)

In [55]:
lsoa_loc_pop_depr.isnull().sum()

lsoa_code          0
date               0
pfa_main           0
crime_count        0
avg_crime_sev      0
lsoa_name        206
lad_code         206
lad_name         206
pfa_code         206
pfa_name         206
year               0
population       206
imd_score        210
incm_score       210
empl_score       210
edcn_score       210
hous_score       210
dtype: int64

In [56]:
lsoa_loc_pop_depr = lsoa_loc_pop_depr.dropna()

In [57]:
lsoa_loc_pop_depr.duplicated().sum()

np.int64(0)

In [58]:
# Export to csv file

## Output Final Table to csv
lsoa_loc_pop_depr.to_csv('../Data/Output/crime-final-lsoa.csv', index=False)

print(f'crime-final-lsoa.csv File successfully created: {Path('../Data/Output/crime-final-lsoa.csv').exists()}')

crime-final-lsoa.csv File successfully created: True


## PFA Grained Data

Grained / year
Final Columns:  
PFA columns | PFA main | Date | Year | Total Population | Total Crime Count | Avg Crime per 1000 / month | Avg Crime Severity || avg deprivation stats ||

In [59]:
# Formulate final table
by_pfa = (
    lsoa_loc_pop_depr
    .groupby(['pfa_main', 'date'], as_index=False)
    .agg(
        pfa_code=('pfa_code', lambda x: x.mode().iloc[0]),
        year=('year', 'first'),
        total_pop=('population', 'sum'),
        crime_count=('crime_count', 'sum'),
        avg_crime_sev=('avg_crime_sev', 'mean'),
        avg_imd_score=('imd_score', 'mean'),
        avg_incm_score=('incm_score', 'mean'),
        avg_empl_score=('empl_score', 'mean'),
        avg_edcn_score=('edcn_score', 'mean'),
        avg_hous_score=('hous_score', 'mean')
    )
)

by_pfa['crime_count_per_1000']=(by_pfa['crime_count'] / by_pfa['total_pop']) *1000

by_pfa.sample(7)

,pfa_main,date,pfa_code,year,total_pop,crime_count,avg_crime_sev,avg_imd_score,avg_incm_score,avg_empl_score,avg_edcn_score,avg_hous_score,crime_count_per_1000
102,West Midlands Police,2025-10-01,E23000014,2025,3077098.0,26756,380.521595,31.346432,0.345212,0.190033,31.228096,19.258232,8.695206
83,West Midlands Police,2024-03-01,E23000014,2024,3053526.0,27910,372.432034,31.206558,0.343712,0.189224,31.097583,19.251733,9.140253
29,Merseyside Police,2025-09-01,E23000004,2025,1483918.5,12432,382.813812,31.509469,0.296137,0.202164,29.033205,11.898192,8.377819
69,South Yorkshire Police,2026-01-01,E23000011,2026,1485829.0,12133,337.293217,28.924634,0.283244,0.176674,31.777221,17.453237,8.165812
1,Merseyside Police,2023-05-01,E23000004,2023,1461102.0,14503,385.104087,31.490811,0.295970,0.202201,29.007488,11.867020,9.926070
27,Merseyside Police,2025-07-01,E23000004,2025,1482269.5,13078,389.320803,31.540465,0.296657,0.202435,29.110808,11.767940,8.822957
130,West Yorkshire Police,2025-02-01,E23000010,2025,2452915.5,21431,398.304766,27.802202,0.275085,0.161095,28.883942,17.376600,8.736950
